# Notebook 05 — Support Vector Machines

**Module:** 13 — Machine Learning for Biology  
**Tier:** 1 — Master  
**Notebook:** 05 of 15  
**Time estimate:** 90 minutes

> SVMs dominated bioinformatics in the 2000s and are still best-in-class for
> small-sample, high-dimensional problems — exactly the setting of most proteomics
> and genomics datasets. The kernel trick is one of the most beautiful ideas in ML.

---
## Step 1 — Motivation

Protein sequence classification (splice site detection, signal peptide prediction,
transcription factor binding), microarray cancer classification, and peptide-MHC
binding prediction were all dominated by SVM approaches before deep learning.
Understanding SVMs deeply gives intuition for margins, kernels, and the geometry
of high-dimensional feature spaces that is essential for understanding deep learning.

---
## Step 2 — Intuition

**Maximum margin classifier:** Among all hyperplanes that correctly separate the
classes, find the one that is farthest from all training points.

The margin is $\frac{2}{\|\mathbf{w}\|}$. Maximizing the margin is equivalent to
minimizing $\|\mathbf{w}\|^2$.

**Support vectors:** The training points that lie exactly on the margin boundary.
The solution depends only on these points — all others can be removed and the
hyperplane won't change.

**Soft margin (C-SVM):** Allow some misclassifications (slack variables $\xi_i$)
controlled by hyperparameter $C$:
- Large $C$: low tolerance for misclassification → small margin → overfits
- Small $C$: allows more misclassification → large margin → underfits

**Kernel trick:** If data is not linearly separable in input space, map to a higher-
dimensional feature space $\phi(\mathbf{x})$ where it IS separable.
The key insight: we never need $\phi(\mathbf{x})$ explicitly — only the dot product
$K(\mathbf{x}, \mathbf{x}') = \phi(\mathbf{x})^T \phi(\mathbf{x}')$ (the kernel function).

Common kernels:
- **Linear:** $K(x,x') = x^T x'$
- **RBF/Gaussian:** $K(x,x') = \exp(-\gamma \|x-x'\|^2)$ — infinite-dimensional feature space!
- **String kernel (biology):** $K(s,s') = $ (number of shared substrings) — operates on sequences directly

---
## Step 3 — Biological Background

**Protein sequence classification:**
- Input: amino acid sequence (e.g., 9-mer peptide)
- Features: one-hot encoding, k-mer frequency, physicochemical properties
- String kernels operate directly on sequences without explicit feature vectors

**MHC binding prediction (NetMHC):**
- SVM with RBF kernel on peptide sequence features
- Predicts whether a peptide will be presented on MHC class I (→ T-cell response)
- Clinical relevance: vaccine design, cancer neoantigen prediction

**Splice site detection:**
- Classify genomic positions as splice donor/acceptor or non-splice
- Context: 9-nt window around each position → one-hot features → SVM

**Why SVMs work well for small n:**
SVMs maximize the margin → directly minimizes the VC dimension bound:
$\text{test error} \leq \text{train error} + \sqrt{\frac{h(\log(2n/h)+1) - \log(\delta/4)}{n}}$
where $h$ is the VC dimension. Larger margin → smaller effective $h$.

---
## Step 4 — Mathematical Explanation

**Hard-margin primal:**
$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 \quad \text{s.t.} \quad y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1 \; \forall i$$

**Lagrangian dual:**
$$\max_{\alpha \geq 0} \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j \mathbf{x}_i^T\mathbf{x}_j \quad \text{s.t.} \sum_i \alpha_i y_i = 0$$

At solution: $\mathbf{w}^* = \sum_i \alpha_i y_i \mathbf{x}_i$ (weighted sum of support vectors only).

**KKT conditions:** $\alpha_i > 0$ only if $y_i(\mathbf{w}^T\mathbf{x}_i + b) = 1$ (on the margin).
All other $\alpha_i = 0$ — the solution is sparse in training points.

**Kernel substitution:** Replace $\mathbf{x}_i^T\mathbf{x}_j$ with $K(\mathbf{x}_i, \mathbf{x}_j)$ everywhere.
Decision function: $f(\mathbf{x}) = \sum_i \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b$.

**Soft margin ($C$-SVM):**
$$\min \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_i \xi_i \quad \text{s.t.} \quad y_i(\mathbf{w}^T\mathbf{x}_i + b) \geq 1 - \xi_i, \; \xi_i \geq 0$$

Dual: same as hard-margin but $0 \leq \alpha_i \leq C$.

In [ ]:
# Step 6 — Python: SVM analysis using sklearn + manual margin visualization
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline

# ---- Generate protein binding dataset ----
# Simulate peptide-MHC binding: 200 peptides, 5 physicochemical features
# Binders tend to have: short length, hydrophobic anchor, specific charge
rng = np.random.default_rng(42)
n = 300

def make_binding_data(n=300, seed=42):
    rng = np.random.default_rng(seed)
    # Features: hydrophobicity, charge, length_score, anchor_match, mw
    # Binders: high hydrophobicity, moderate charge, positive anchor
    X_bind = rng.multivariate_normal([2, 0.5, 1, 1, 0],
                                        [[0.6, 0.1, 0, 0.2, 0],
                                         [0.1, 0.4, 0, 0, 0],
                                         [0, 0, 0.5, 0, 0],
                                         [0.2, 0, 0, 0.6, 0],
                                         [0, 0, 0, 0, 0.5]], n//3)
    # Non-binders: variable features
    X_non = rng.multivariate_normal([0, -0.5, -1, -1, 0],
                                      [[1.0, 0, 0, 0, 0],
                                       [0, 0.8, 0, 0, 0],
                                       [0, 0, 0.8, 0, 0],
                                       [0, 0, 0, 0.8, 0],
                                       [0, 0, 0, 0, 0.8]], 2*n//3)
    X = np.vstack([X_bind, X_non])
    y = np.array([1]*(n//3) + [-1]*(2*n//3))  # SVM uses -1/+1
    return X, y

X, y = make_binding_data(300)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                       random_state=42, stratify=y)

# ---- Compare Linear, RBF, and Polynomial SVM kernels ----
kernels = ['linear', 'rbf', 'poly']
print('SVM kernel comparison (5-fold CV, balanced accuracy):')
for kernel in kernels:
    pipe = make_pipeline(StandardScaler(), SVC(kernel=kernel, class_weight='balanced'))
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='balanced_accuracy')
    print(f'  {kernel:<10}: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# ---- Hyperparameter tuning: RBF SVM on C and gamma ----
pipe_rbf = make_pipeline(StandardScaler(), SVC(kernel='rbf', class_weight='balanced'))
param_grid = {'svc__C': [0.01, 0.1, 1, 10, 100], 'svc__gamma': ['scale', 0.01, 0.1, 1]}
grid_search = GridSearchCV(pipe_rbf, param_grid, cv=5, scoring='balanced_accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
print(f'\nBest RBF SVM params: {grid_search.best_params_}')
print(f'Best CV score: {grid_search.best_score_:.4f}')
print(f'Test balanced accuracy: {grid_search.score(X_test, y_test):.4f}')

# ---- Support vectors ----
best_pipe = grid_search.best_estimator_
best_svc = best_pipe.named_steps['svc']
print(f'\nNumber of support vectors: {best_svc.n_support_}')
print(f'As fraction of training set: {best_svc.n_support_.sum() / len(X_train):.2f}')

In [ ]:
# Step 7 — Visualization (2D projection for visualization)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Use first 2 PCA components for visualization
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_train_sc = StandardScaler().fit_transform(X_train)
X_2d = pca.fit_transform(X_train_sc)

# Train 2D SVM for visualization
for ax, (kernel, C_val) in zip(axes, [('linear', 1), ('rbf', 1), ('poly', 1)]):
    svm_2d = SVC(kernel=kernel, C=C_val, class_weight='balanced')
    svm_2d.fit(X_2d, y_train)
    
    xx, yy = np.meshgrid(np.linspace(X_2d[:,0].min()-0.5, X_2d[:,0].max()+0.5, 200),
                           np.linspace(X_2d[:,1].min()-0.5, X_2d[:,1].max()+0.5, 200))
    Z = svm_2d.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.6)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors=['blue', 'black', 'red'],
                 linestyles=['--', '-', '--'], linewidths=[0.8, 1.5, 0.8])
    
    # Support vectors
    sv_idx = svm_2d.support_
    ax.scatter(X_2d[y_train==1, 0], X_2d[y_train==1, 1],
                 c='tomato', s=15, alpha=0.5, label='Binder')
    ax.scatter(X_2d[y_train==-1, 0], X_2d[y_train==-1, 1],
                 c='steelblue', s=15, alpha=0.5, label='Non-binder')
    ax.scatter(X_2d[sv_idx, 0], X_2d[sv_idx, 1],
                 s=80, facecolors='none', edgecolors='black', lw=1.5, label=f'SVs (n={len(sv_idx)})')
    
    acc_2d = svm_2d.score(X_2d, y_train)
    ax.set_title(f'{kernel.upper()} kernel\n(2D PCA, C={C_val}, train acc={acc_2d:.3f})', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

plt.suptitle('Module 13 NB05: Support Vector Machines\n(margin boundaries + support vectors)',
               fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('svm.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Show the effect of `C` on the SVM decision boundary: train with `C=0.001`,
   `C=1`, `C=1000` on the 2D PCA data. How does the margin width change?
2. For the RBF kernel `K(x,x') = exp(-γ||x-x'||²)`, what does it mean when
   `γ` is very large? What does the decision function look like?
3. Implement a string kernel: `K(s,s') = sum of counts of shared k-mers` for k=3,
   on a set of 9-mer peptide sequences.
4. Why does the SVM solution depend only on support vectors? Prove from KKT conditions.

---
## Step 10 — Quiz

1. What is the maximum margin hyperplane? What are support vectors?
2. What is the soft margin SVM? What does hyperparameter C control?
3. What is the kernel trick? Why don't we need φ(x) explicitly?
4. What does the RBF kernel K(x,x') = exp(-γ||x-x'||²) measure geometrically?
5. Why are SVMs well-suited for small-n, high-p biological problems?

---
## Step 12 — Reflection

> *[In one paragraph: explain the kernel trick in plain language. Why can the RBF
> kernel represent an infinite-dimensional feature space, and why does this allow
> it to classify non-linearly separable data?]*

---
*Next: `06_k_nearest_neighbors_and_naive_bayes.ipynb`*